# 3. Матрица, правые части и бесконечный свободный хвост

Это основной вычислительный практикум. Мы печатаем все элементы маленькой
матрицы, решаем её общим методом и сопоставляем с специализированной прогонкой.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium
from lighthit.angular import (free_moments_and_tail, solve_tail_system,
                              dense_finite_rank_reference, _free_moments_and_ratios)

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())
K_PER_M, OMEGA_PER_NS = 0.2, 0.0
d0 = medium.extinction_per_m - 1j*OMEGA_PER_NS/medium.speed_m_per_ns
print('d0 =', d0)


## 3.1. Ортонормированный базис и рекурсия стриминга

$$p_\ell=\sqrt{\tfrac{2\ell+1}{2}}P_\ell,\qquad \int p_\ell p_j\,d\mu=\delta_{\ell j},
\qquad \int p_\ell\,d\mu=\sqrt2\,\delta_{\ell0}.$$
Умножение на $\mu$ связывает только соседние степени:
$$\mu p_\ell=a_\ell p_{\ell-1}+a_{\ell+1}p_{\ell+1},\qquad a_\ell=\ell/\sqrt{4\ell^2-1}.$$
Обе формулы проверяем квадратурой, а не принимаем на веру.


In [ ]:
from scipy.special import eval_legendre, roots_legendre

mu_q, w_q = roots_legendre(1024)
ell = np.arange(10)
pl = eval_legendre(ell[:, None], mu_q)*np.sqrt((2*ell+1)/2)[:, None]
a = np.where(ell > 0, ell/np.sqrt(np.where(ell > 0, 4*ell**2 - 1, 1)), 0.)

print('ортонормированность:', np.max(np.abs((pl*w_q) @ pl.T - np.eye(len(ell)))))
print('проекция единицы   :', np.max(np.abs(pl @ w_q - np.sqrt(2)*(ell == 0))))
worst = max(np.max(np.abs(mu_q*pl[l] - a[l]*pl[l-1] - a[l+1]*pl[l+1]))
            for l in range(1, len(ell)-1))
print('рекурсия стриминга :', worst)
print('a_1..a_3 =', a[1:4], ' против', [1/np.sqrt(3), 2/np.sqrt(15), 3/np.sqrt(35)])
assert worst < 1e-12


## 3.1. Каждая степень задаёт строку

Разложим $h=\sum h_\ell p_\ell$. Рекурсия
$\mu p_\ell=a_\ell p_{\ell-1}+a_{\ell+1}p_{\ell+1}$,
$a_\ell=\ell/\sqrt{4\ell^2-1}$, $a_0=0$, даёт
$$ik a_\ell h_{\ell-1}+(d_0-\gamma_\ell)h_\ell+ik a_{\ell+1}h_{\ell+1}
=\sqrt2\,\delta_{\ell0}.$$

Первые строки:
$$
(\mu_a-i\omega/v)h_0+\frac{ik}{\sqrt3}h_1=\sqrt2,
$$
$$\frac{ik}{\sqrt3}h_0+(d_0-\gamma_1)h_1+\frac{2ik}{\sqrt{15}}h_2=0,$$
$$\frac{2ik}{\sqrt{15}}h_1+(d_0-\gamma_2)h_2+\frac{3ik}{\sqrt{35}}h_3=0.$$

В третьей строке есть $h_3$: матрица пока бесконечна. Степень $L$ задаёт
$\gamma_{\ell>L}=0$, но не зануляет $h_{\ell>L}$.

In [ ]:
L = 3
b, tail = free_moments_and_tail(np.array([K_PER_M]), d0, L)
b, rho = b[0], tail[0]
aa = np.arange(1, L+2)/np.sqrt(4*np.arange(1, L+2)**2 - 1)
gamma = medium.scattering_per_m*medium.g**np.arange(L+1)

M = (np.diag(d0 - gamma).astype(complex)
     + np.diag(1j*K_PER_M*aa[:-1], 1) + np.diag(1j*K_PER_M*aa[:-1], -1))
M[-1, -1] += 1j*K_PER_M*aa[-1]*rho          # точное исключение хвоста
f = np.zeros(L+1, complex); f[0] = np.sqrt(2)

print('a =', aa); print('gamma =', gamma); print('r_L =', rho)
print('M =\n', M)
print('строка 0 диагональ =', M[0, 0], ' mu_a =', medium.absorption_per_m)
print('симметрия:', np.max(np.abs(M - M.T)), ' эрмитовость:', np.max(np.abs(M - M.conj().T)))
h = np.linalg.solve(M, f)
print('h =', h)


## 3.2. Исключение хвоста

Выше $L$ правая часть равна нулю. Допустимое затухающее решение свободной
рекурсии пропорционально $b_\ell$, поэтому $h_{L+1}=r_Lh_L$,
$r_L=b_{L+1}/b_L$. Конечная матрица:
$$
M_{\ell j}=(d_0-\gamma_\ell)\delta_{\ell j}
+ik a_\ell\delta_{j,\ell-1}+ik a_{\ell+1}\delta_{j,\ell+1}
+ik a_{L+1}r_L\delta_{\ell L}\delta_{jL}.
$$
Строки и столбцы $0,\ldots,L$. Последнее слагаемое — весь свободный хвост.
Матрица комплексно-симметрична, но обычно не эрмитова.

In [ ]:
# Точный хвост против жёсткого обрыва, оба против независимого плотного контроля.
rows = []
for degree in (3, 8, 16, 32):
    bb, tt = free_moments_and_tail(np.array([K_PER_M]), d0, degree)
    g = medium.scattering_per_m*medium.g**np.arange(degree+1)
    c = np.arange(1, degree+2)/np.sqrt(4*np.arange(1, degree+2)**2 - 1)
    base = (np.diag(d0 - g).astype(complex)
            + np.diag(1j*K_PER_M*c[:-1], 1) + np.diag(1j*K_PER_M*c[:-1], -1))
    rhs = np.zeros(degree+1, complex); rhs[0] = np.sqrt(2)
    exact = base.copy(); exact[-1, -1] += 1j*K_PER_M*c[-1]*tt[0]
    control = dense_finite_rank_reference(K_PER_M, OMEGA_PER_NS, medium, degree,
                                          degree, quadrature_order=1024)
    scale = np.max(np.abs(control))
    rows.append((degree,
                 np.max(np.abs(np.linalg.solve(exact, rhs) - control))/scale,
                 np.max(np.abs(np.linalg.solve(base, rhs) - control))/scale))
display(Markdown('| L | точный хвост | жёсткий обрыв |\n|--:|--:|--:|\n'
                 + '\n'.join(f'| {d} | {e:.1e} | {t:.1e} |' for d, e, t in rows)))

# И то же решение через прогонку пакета.
packaged = solve_tail_system(np.array([K_PER_M]), d0, np.array([rho]), gamma, f[None, :])[0]
print('прогонка против плотного solve:', np.max(np.abs(packaged - h))/np.max(np.abs(h)))
assert np.max(np.abs(packaged - h))/np.max(np.abs(h)) < 1e-12


## 3.4. Минимальная ветвь, прямая неустойчивость и Miller

Рекурсия $F_{\ell+1}=\frac{(2\ell+1)zF_\ell-\ell F_{\ell-1}}{\ell+1}$, $z=id_0/k$,
имеет две ветви: $Q_\ell(z)$ затухает, $P_\ell(z)$ растёт. Моменты идут по $Q$.
Прямой ход подмешивает растущую ветвь на каждом шаге, и при сильном затухании
результат состоит почти целиком из неё. Обратный ход
$$R_{\ell-1}=\frac{\ell}{(2\ell+1)z-(\ell+1)R_\ell}$$
забывает затравку. Ниже — обе ветви на одном узле с сильным затуханием.


In [ ]:
k_small = 1e-2
z = 1j*d0/k_small
N = 60

# Прямой ход из точных F_0, F_1.
F = np.zeros(N+2, complex)
F[0] = 2*np.arctan(k_small/d0)/k_small
F[1] = z*F[0] + 2/(1j*k_small)
for l in range(1, N+1):
    F[l+1] = ((2*l+1)*z*F[l] - l*F[l-1])/(l+1)
forward_ratio = F[1:]/F[:-1]

# Обратный ход с затравкой ноль, из большой глубины.
R = 0j
backward = np.zeros(N+1, complex)
for l in range(N+1+400, 0, -1):
    R = l/((2*l+1)*z - (l+1)*R)
    if l-1 <= N:
        backward[l-1] = R

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.semilogy(np.abs(forward_ratio[:N+1] - backward)/np.abs(backward))
ax.set(xlabel='степень', ylabel='отн. расхождение прямого и обратного хода',
       title=f'k = {k_small} 1/m')
plt.show()

_, packaged_tail = free_moments_and_tail(np.array([k_small]), d0, N)
norm = np.sqrt((2*N+3)/(2*N+1))*backward[N]
print('пакет против независимой глубокой дроби:', abs(packaged_tail[0] - norm)/abs(norm))
assert abs(packaged_tail[0] - norm)/abs(norm) < 1e-12


## 3.5. $k=0$ — отдельная ветвь, а не предел

При $k=0$ множитель теряет зависимость от направления:
$$b_0=\frac{\sqrt2}{d_0},\qquad b_{\ell>0}=0,\qquad r_\ell=0,$$
матрица становится диагональной. Это нули точные, а не маленькие числа:
формулы для $z$ и $F_0$ делят на $k$, и подстановка $k=10^{-15}$ вместо ветви
была бы арифметикой на краю диапазона без всякой нужды.


In [ ]:
b0, tail0 = free_moments_and_tail(np.array([0.0]), d0, 8)
print('b_0 =', b0[0, 0], ' ожидание', np.sqrt(2)/d0)
print('max |b_l>0| =', np.max(np.abs(b0[0, 1:])), ' max |r| =', abs(tail0[0]))
assert b0[0, 0] == np.sqrt(2)/d0 and np.max(np.abs(b0[0, 1:])) == 0.0

# Та же ветвь внутри батча: нулевой узел не портит остальные.
bb, rr = _free_moments_and_ratios(np.array([0.0, 0.05, 0.5, 5.0]), d0, 8)
print('конечность всего батча:', np.isfinite(bb).all() and np.isfinite(rr).all())


## Задания

1. Повторить печать матрицы для $L=2$ и $\omega\neq0$; проследить, где
появляется мнимая часть диагонали.
2. Вывести исключение хвоста как дополнение Шура и показать, что поправка
имеет ранг единица.
3. Оценить по таблице, сколько лишних степеней нужно жёсткому обрыву, чтобы
догнать точный хвост при $L=8$.
4. Написать свою прогонку и сравнить не только решение, но и невязку $Mh-f$.
5. Различать степень рассеяния $L$, выходную степень $J$ и число столкновений;
показать, что размер системы от $J$ не зависит.
6. Проверить, при каком $k$ правило $\eta(N+3)<3$ переключает ветвь, и
измерить, как глубина старта растёт с $N$.

Код: `free_moments_and_tail`, `solve_tail_system`, `dense_finite_rank_reference`.
